# Module 5 — Batch Processing with Apache Spark
## Notebook 1: Spark Basics

**Topics covered:**
- SparkSession setup (local classic mode)
- Reading Parquet files
- Schema inspection
- DataFrame transformations (select, filter, withColumn)
- Repartitioning
- Writing Parquet output
- Homework Q1 & Q2

**Dataset:** Yellow Taxi October 2024  
**Source:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

## 1. Start SparkSession

In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, TimestampType
)

print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.1.1


In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("dezoomcamp-module5-basics") \
    .config("spark.api.mode", "classic") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Reduce log noise in notebook output
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version  : {spark.version}")
print(f"Default cores  : {spark.sparkContext.defaultParallelism}")

Spark version  : 4.1.1
Default cores  : 2


## 2. Read Parquet File

Parquet is a **columnar** binary format — Spark reads only the columns it needs, making queries on wide tables very efficient.

Key point: reading is **lazy** — nothing actually executes until an action (`.show()`, `.count()`, `.write`) is called.

In [6]:
DATA_PATH = "../data/yellow_tripdata_2024-10.parquet"

df = spark.read.parquet(DATA_PATH)

# .count() is an action — triggers execution
print(f"Row count: {df.count():,}")

UnsupportedOperationException: getSubject is not supported

## 3. Schema Inspection

Unlike pandas, Spark DataFrames are **strongly typed**. The schema is inferred from Parquet metadata — no data is read to do this.

In [ ]:
df.printSchema()

In [ ]:
# Preview first 5 rows — truncate=False shows full column values
df.show(5, truncate=False)

In [ ]:
# Column count and names
print(f"Columns ({len(df.columns)}): {df.columns}")

In [ ]:
# Summary statistics — like pandas .describe()
df.select(
    "trip_distance",
    "fare_amount",
    "passenger_count"
).describe().show()

## 4. Core Transformations

All transformations are **lazy** — they build a logical plan (DAG) but don't execute. Spark optimises the full plan before running anything.

In [ ]:
# Select a subset of columns
df_slim = df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount"
)

df_slim.show(3)

In [ ]:
# Add derived columns with withColumn
df_enriched = df_slim \
    .withColumn("pickup_date",  F.to_date("tpep_pickup_datetime")) \
    .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime")) \
    .withColumn("trip_duration_min",
                F.round(
                    (F.unix_timestamp("tpep_dropoff_datetime") -
                     F.unix_timestamp("tpep_pickup_datetime")) / 60, 2
                ))

df_enriched.select(
    "tpep_pickup_datetime", "pickup_date", "pickup_hour", "trip_duration_min"
).show(5)

In [ ]:
# Filter: valid trips only
df_valid = df_enriched.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount")   > 0) &
    (F.col("trip_duration_min") > 0) &
    (F.col("trip_duration_min") < 300)   # exclude trips > 5 hours
)

print(f"Valid rows: {df_valid.count():,}  "
      f"(dropped {df.count() - df_valid.count():,})")

## 5. Partitioning

A Spark DataFrame is divided into **partitions** — chunks processed in parallel across cores/executors.

- `repartition(n)` — shuffles data, produces exactly `n` evenly-sized partitions (expensive but balanced)
- `coalesce(n)` — reduces partitions without a full shuffle (cheap, but can produce uneven sizes)

Rule of thumb: target **128 MB per partition** for writes.

In [ ]:
print(f"Current partitions: {df.rdd.getNumPartitions()}")

In [ ]:
# Homework Q1: repartition to 4 and write parquet
df_hw = spark.read.parquet(DATA_PATH)   # fresh read, no filters
df_4  = df_hw.repartition(4)

print(f"Partitions after repartition(4): {df_4.rdd.getNumPartitions()}")

## 6. Write Parquet

Writing triggers execution of the full DAG. Each partition becomes one `.parquet` file.

In [ ]:
import os

OUTPUT_PATH = "../data/yellow_oct2024_4partitions"

df_4.write \
    .mode("overwrite") \
    .parquet(OUTPUT_PATH)

print(f"Written to: {OUTPUT_PATH}")

In [ ]:
# Homework Q1: average parquet file size
parquet_files = [
    f for f in os.listdir(OUTPUT_PATH)
    if f.endswith(".parquet")
]

sizes_mb = [
    os.path.getsize(os.path.join(OUTPUT_PATH, f)) / (1024 ** 2)
    for f in parquet_files
]

print(f"Parquet files   : {len(parquet_files)}")
print(f"Sizes (MB)      : {[round(s, 2) for s in sizes_mb]}")
print(f"Average size    : {sum(sizes_mb)/len(sizes_mb):.2f} MB  <-- Homework Q1 answer")

## 7. Homework Q2 — Trips starting on October 15

Filter for trips where `tpep_pickup_datetime` is on the 15th of October 2024.

In [ ]:
df_oct15 = df_hw.filter(
    F.to_date(F.col("tpep_pickup_datetime")) == F.lit("2024-10-15")
)

oct15_count = df_oct15.count()
print(f"Trips on Oct 15: {oct15_count:,}  <-- Homework Q2 answer")

## 8. Quick EDA — Trip patterns

In [ ]:
# Trips per day — understand the full month's distribution
df_hw \
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_date") \
    .count() \
    .orderBy("pickup_date") \
    .show(35)

In [ ]:
# Trips by hour of day
df_hw \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .groupBy("pickup_hour") \
    .count() \
    .orderBy("pickup_hour") \
    .show(24)

In [ ]:
# Vendor breakdown
df_hw.groupBy("VendorID").count().orderBy("VendorID").show()

## 9. Understanding Lazy Evaluation — the DAG

Spark builds a **Directed Acyclic Graph (DAG)** of transformations. Nothing runs until an action triggers it. You can inspect the plan before executing.

In [ ]:
# See the logical + physical execution plan
df_oct15.explain(mode="formatted")

## 10. Stop SparkSession

In [ ]:
spark.stop()
print("SparkSession stopped.")

---
## Summary

| Concept | Key takeaway |
|---|---|
| SparkSession | Entry point; `local[*]` uses all available cores |
| Lazy evaluation | Transformations build a plan; actions execute it |
| Parquet | Columnar format; schema embedded; Spark reads it natively |
| `repartition(n)` | Full shuffle → exactly n equal partitions |
| `coalesce(n)` | Reduces partitions without shuffle (uneven but cheap) |
| `write.parquet()` | Each partition → one `.parquet` file in the output folder |

**Next:** Notebook 2 — Spark SQL & joins (homework Q3/Q4)